<a href="https://colab.research.google.com/github/Zunera-tech/-CodeAlpha_CreditRiskPrediction/blob/main/CodeAlpha_CreditRiskPrediction_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Risk Prediction
## Predicting Financial Distress using Machine Learning
**Dataset:** Give Me Some Credit (Kaggle)  
**Author:** Zunera  
**GitHub:** CodeAlpha_CreditRiskPrediction

---
### Project Goal
Predict whether a borrower will experience **financial distress** (loan default) within 2 years.

###  What makes this project exceptional?
-  Feature Engineering — 8 new meaningful financial features
-  Class Imbalance handling with **SMOTE**
-  4 Models compared — LR, Decision Tree, Random Forest, Gradient Boosting
-  Full Evaluation — Precision, Recall, F1, ROC-AUC
-  **SHAP Explainability** — Why did the model predict default?


In [ ]:
# Installing libraries
!pip install shap imbalanced-learn -q

#  importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score, accuracy_score,
    precision_recall_curve
)
from imblearn.over_sampling import SMOTE
import shap

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("All libraries imported successfully!")
print(f"   Pandas    : {pd.__version__}")
print(f"   Numpy     : {np.__version__}")


---
##  Load Dataset



In [ ]:
#  LOADING DATASET


df = pd.read_csv('cs-training.csv', index_col=0)

print("=" * 50)
print("       DATASET OVERVIEW")
print("=" * 50)
print(f"  Shape            : {df.shape}")
print(f"  Total Borrowers  : {df.shape[0]:,}")
print(f"  Total Features   : {df.shape[1]-1}")
print(f"  Target Column    : SeriousDlqin2yrs")
print("=" * 50)

print("\n Target Distribution:")
counts = df['SeriousDlqin2yrs'].value_counts()
pct    = df['SeriousDlqin2yrs'].value_counts(normalize=True) * 100
print(f"  No Default (0) : {counts[0]:,} ({pct[0]:.1f}%)")
print(f"  Default    (1) : {counts[1]:,} ({pct[1]:.1f}%)")
print(f"\n  Class Imbalance detected! Only {pct[1]:.1f}% defaulters.")
print("   We will fix this with SMOTE in Step 4.")

df.head()


---
##  Step 2 — Exploratory Data Analysis (EDA)


In [ ]:
#Data cleaning
# MISSING VALUES

print("Missing Values:")
missing     = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df  = pd.DataFrame({
    'Missing Count': missing,
    'Missing %'    : missing_pct.round(2)
})
print(missing_df[missing_df['Missing Count'] > 0])

# Visual
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class Distribution
axes[0].bar(['No Default (0)', 'Default (1)'],
            [counts[0], counts[1]],
            color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)
axes[0].set_title('Target Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 500, f'{v:,}\n({pct[i]:.1f}%)',
                 ha='center', fontweight='bold')

# Missing Values
missing_nonzero = missing_pct[missing_pct > 0]
axes[1].barh(missing_nonzero.index, missing_nonzero.values,
             color='#e74c3c', edgecolor='black')
axes[1].set_title('Missing Values (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Missing %')
for i, v in enumerate(missing_nonzero.values):
    axes[1].text(v + 0.2, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved!")


In [ ]:
#FEATURE DISTRIBUTIONS
key_features = [
    'age', 'DebtRatio', 'MonthlyIncome',
    'RevolvingUtilizationOfUnsecuredLines',
    'NumberOfTimes90DaysLate',
    'NumberOfTime30-59DaysPastDueNotWorse'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, feat in zip(axes.flatten(), key_features):
    default     = df[df['SeriousDlqin2yrs'] == 1][feat].dropna()
    no_default  = df[df['SeriousDlqin2yrs'] == 0][feat].dropna()

    # Clip extreme outliers for visualization
    upper = df[feat].quantile(0.98)
    default    = default.clip(upper=upper)
    no_default = no_default.clip(upper=upper)

    ax.hist(no_default, bins=40, alpha=0.6,
            color='#2ecc71', label='No Default', density=True)
    ax.hist(default,    bins=40, alpha=0.6,
            color='#e74c3c', label='Default',    density=True)
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_ylabel('Density')

plt.suptitle('Feature Distributions by Target Class',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# CORRELATION HEATMAP
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5,
            annot_kws={'size': 8}, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key Correlations with Target (SeriousDlqin2yrs):")
target_corr = corr['SeriousDlqin2yrs'].drop('SeriousDlqin2yrs').sort_values(ascending=False)
print(target_corr.round(3))


---
## Step 3 — Feature Engineering



In [ ]:
#FEATURE ENGINEERING
def engineer_features(df):
    df = df.copy()

    #  Handling  Missing Values
    df['MonthlyIncome'].fillna(df['MonthlyIncome'].median(), inplace=True)
    df['NumberOfDependents'].fillna(0, inplace=True)

    # Remove  outliers
    df['age']      = df['age'].clip(18, 100)
    df['DebtRatio'] = df['DebtRatio'].clip(0, 20)
    df['MonthlyIncome'] = df['MonthlyIncome'].clip(
        upper=df['MonthlyIncome'].quantile(0.99))
    df['RevolvingUtilizationOfUnsecuredLines'] =         df['RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)

    # Total Late Payments
    df['TotalLatePayments'] = (
        df['NumberOfTime30-59DaysPastDueNotWorse'] +
        df['NumberOfTime60-89DaysPastDueNotWorse'] +
        df['NumberOfTimes90DaysLate']
    )

    #   Income per Dependent
    df['IncomePerDependent'] = (
        df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)
    )

    #  Monthly Debt Obligation estimate
    df['MonthlyDebtObligation'] = df['DebtRatio'] * df['MonthlyIncome']

    # High Debt Ratio flag (> 0.43 = risky)
    df['HighDebtRatio'] = (df['DebtRatio'] > 0.43).astype(int)

    # High Credit Utilization (> 0.8 = danger zone)
    df['HighCreditUtilization'] = (
        df['RevolvingUtilizationOfUnsecuredLines'] > 0.8).astype(int)

    #Has Ever Been Severely Delinquent (90+ days)
    df['HasSevereDelinquency'] = (
        df['NumberOfTimes90DaysLate'] > 0).astype(int)

    # Loan Burden Score
    df['LoanBurdenScore'] = (
        df['NumberOfOpenCreditLinesAndLoans'] * df['DebtRatio']
    )

    #  Age Risk Group
    df['YoungBorrower'] = (df['age'] < 30).astype(int)

    return df


df_eng = engineer_features(df)

print(f"Original features : {df.shape[1] - 1}")
print(f"After engineering : {df_eng.shape[1] - 1}")
print(f"New features added: {df_eng.shape[1] - df.shape[1]}")
print()
print("New features created:")
new_feats = ['TotalLatePayments', 'IncomePerDependent',
             'MonthlyDebtObligation', 'HighDebtRatio',
             'HighCreditUtilization', 'HasSevereDelinquency',
             'LoanBurdenScore', 'YoungBorrower']
for f in new_feats:
    print(f"   {f}")


---
Preprocessing + Handling Class Imbalance (SMOTE)




In [ ]:
#  PREPROCESSING

# Define features
FEATURE_COLS = [c for c in df_eng.columns
                if c != 'SeriousDlqin2yrs']
TARGET       = 'SeriousDlqin2yrs'

X = df_eng[FEATURE_COLS]
y = df_eng[TARGET]

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}")
print(f"Features used    : {len(FEATURE_COLS)}")

# ── Train-Test Split (Stratified!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # Important for imbalanced data!
)

print(f"\nTrain size : {X_train.shape[0]:,}")
print(f"Test  size : {X_test.shape[0]:,}")

# ── Feature Scaling
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

# ── SMOTE — Handle Class Imbalance
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print(f"\nBefore SMOTE:")
print(f"  No Default: {(y_train==0).sum():,}")
print(f"  Default   : {(y_train==1).sum():,}")

print(f"\nAfter SMOTE (balanced!):")
print(f"  No Default: {(y_train_res==0).sum():,}")
print(f"  Default   : {(y_train_res==1).sum():,}")

# Convert back for easier handling
X_train_sc_orig = X_train_sc.copy()  # Keep original (non-SMOTE) for some models


---
##   Model Training
Training **4 classification models** and comparing them.


In [ ]:
#  TRAIN ALL MODELS


models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        C=0.1,
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=50,
        class_weight='balanced',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=20,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        random_state=42
    )
}

results = {}

print("Training models...\n")

for name, model in models.items():
    print(f" Training {name}...", end='')

    # Train
    model.fit(X_train_res, y_train_res)

    # Predict
    y_pred      = model.predict(X_test_sc)
    y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

    # Metrics
    results[name] = {
        'model'    : model,
        'y_pred'   : y_pred,
        'y_prob'   : y_pred_prob,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1-Score' : f1_score(y_test, y_pred),
        'ROC-AUC'  : roc_auc_score(y_test, y_pred_prob)
    }

    print(f" Done! ROC-AUC={results[name]['ROC-AUC']:.4f}")

print("\n All models trained!")


---
## Step 6 — Model Evaluation


In [ ]:

# METRICS TABLE

metrics_df = pd.DataFrame({
    name: {
        'Accuracy' : f"{res['Accuracy']:.4f}",
        'Precision': f"{res['Precision']:.4f}",
        'Recall'   : f"{res['Recall']:.4f}",
        'F1-Score' : f"{res['F1-Score']:.4f}",
        'ROC-AUC'  : f"{res['ROC-AUC']:.4f}",
    }
    for name, res in results.items()
}).T

print("=" * 70)
print("                MODEL PERFORMANCE COMPARISON")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

best_model_name = max(results, key=lambda x: results[x]['ROC-AUC'])
print(f"\n Best Model: {best_model_name}")
print(f"   ROC-AUC = {results[best_model_name]['ROC-AUC']:.4f}")


In [ ]:
#  ROC CURVES
plt.figure(figsize=(10, 6))
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, linewidth=2.5, color=color,
             label=f"{name} (AUC = {res['ROC-AUC']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
plt.fill_between([0,1], [0,1], alpha=0.05, color='gray')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models Comparison',
          fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────
#  STEP 6C: MODEL COMPARISON BAR CHART
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_names = ['ROC-AUC', 'F1-Score', 'Precision', 'Recall']
model_names  = list(results.keys())
x = np.arange(len(model_names))
width = 0.2
colors_bar = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

# Left: All metrics grouped
for i, (metric, color) in enumerate(zip(metric_names, colors_bar)):
    vals = [results[m][metric] for m in model_names]
    axes[0].bar(x + i*width, vals, width, label=metric,
                color=color, edgecolor='black', alpha=0.85)

axes[0].set_xticks(x + width*1.5)
axes[0].set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=10)
axes[0].set_title('All Metrics by Model', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Right: ROC-AUC only (cleaner)
roc_vals = [results[m]['ROC-AUC'] for m in model_names]
bars = axes[1].bar(model_names, roc_vals,
                   color=colors_bar, edgecolor='black', alpha=0.85)
axes[1].set_title('ROC-AUC Comparison', fontsize=13, fontweight='bold')
axes[1].set_ylabel('ROC-AUC Score')
axes[1].set_ylim(0, 1.05)
axes[1].set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=10)
for bar, val in zip(bars, roc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────
#  STEP 6D: CONFUSION MATRIX (Best Model)
# ─────────────────────────────────────────────
best_res = results[best_model_name]
cm       = confusion_matrix(y_test, best_res['y_pred'])
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted\nNo Default', 'Predicted\nDefault'],
            yticklabels=['Actual\nNo Default', 'Actual\nDefault'],
            linewidths=2, linecolor='white', annot_kws={'size': 16})
axes[0].set_title(f'Confusion Matrix — {best_model_name}',
                  fontsize=13, fontweight='bold')

# Classification Report Visual
metrics_vis = {
    'Precision': best_res['Precision'],
    'Recall'   : best_res['Recall'],
    'F1-Score' : best_res['F1-Score'],
    'ROC-AUC'  : best_res['ROC-AUC'],
    'Accuracy' : best_res['Accuracy'],
}
bars = axes[1].barh(list(metrics_vis.keys()),
                    list(metrics_vis.values()),
                    color=['#3498db','#2ecc71','#9b59b6','#e74c3c','#f39c12'],
                    edgecolor='black')
axes[1].set_xlim(0, 1.1)
axes[1].set_title(f'Evaluation Metrics — {best_model_name}',
                  fontsize=13, fontweight='bold')
for bar, val in zip(bars, metrics_vis.values()):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest Model: {best_model_name}")
print(f"  True  Positives (TP) : {tp:,}  → Correctly predicted DEFAULT")
print(f"  True  Negatives (TN) : {tn:,}  → Correctly predicted NO DEFAULT")
print(f"  False Positives (FP) : {fp:,}  → Said default, was actually fine")
print(f"  False Negatives (FN) : {fn:,}  → Missed actual defaults ← most costly!")


In [ ]:
# ─────────────────────────────────────────────
#  STEP 6E: CLASSIFICATION REPORT
# ─────────────────────────────────────────────
print(f"\nDetailed Classification Report — {best_model_name}")
print("="*60)
print(classification_report(
    y_test,
    best_res['y_pred'],
    target_names=['No Default (0)', 'Default (1)']
))


---
##  SHAP Explainability


In [ ]:
#  STEP 7: SHAP EXPLAINABILITY

shap_model_name = 'Random Forest'
shap_model      = results[shap_model_name]['model']

print(f"Running SHAP analysis on: {shap_model_name}")
print("This may take 1-2 minutes...")

# SHAP Explainer (TreeSHAP — fast for tree models)
explainer   = shap.TreeExplainer(shap_model)
n_samples   = 500   # use 500 test samples
X_shap      = X_test_sc[:n_samples]

shap_values = explainer.shap_values(X_shap)

print("✅ SHAP values computed!")


In [ ]:
# SHAP SUMMARY PLOT (Beeswarm)

# Select class 1 (Default) safely for any SHAP version
if isinstance(shap_values, list):
    sv = shap_values[1]              # old SHAP: list of [class0, class1]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]        # new SHAP: (samples, features, classes)
else:
    sv = shap_values                 # already 2D (e.g. XGBoost/LightGBM binary)

print("SHAP values shape:", sv.shape, "| Data shape:", X_shap.shape)

plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv,
    X_shap,
    feature_names=FEATURE_COLS,
    show=False,
    plot_size=(10, 7)
)
plt.title('SHAP Summary — Feature Impact on Default Prediction',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHow to read:")
print("  • Red  = high feature value")
print("  • Blue = low  feature value")
print("  • Right side (positive SHAP) = pushes toward DEFAULT")
print("  • Left  side (negative SHAP) = pushes toward NO DEFAULT")


In [ ]:
#  SHAP BAR PLOT (Mean Importance)

if isinstance(shap_values, list):
    sv = shap_values[1]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]
else:
    sv = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(
    sv,
    X_shap,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    show=False,
    plot_size=(10, 6)
)
plt.title('Top Features by Mean |SHAP Value| — Default Prediction',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_bar_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if isinstance(shap_values, list):
    sv = shap_values[1]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]
else:
    sv = shap_values

# Base value for class 1, safe across SHAP versions / model types
ev = np.atleast_1d(explainer.expected_value)
base_value = ev[1] if len(ev) > 1 else ev[0]

default_indices = np.where(
    results[shap_model_name]['y_pred'][:n_samples] == 1)[0]

if len(default_indices) > 0:
    i = default_indices[0]   # first predicted defaulter

    print(f"Explaining prediction for Test Sample #{i}")
    print(f"  Actual   : {'DEFAULT' if y_test.iloc[i]==1 else 'No Default'}")
    print(f"  Predicted: {'DEFAULT' if results[shap_model_name]['y_pred'][i]==1 else 'No Default'}")
    print(f"  Probability of Default: {results[shap_model_name]['y_prob'][i]:.2%}\n")

    # Works whether X_shap is a NumPy array or a DataFrame
    x_row = X_shap.iloc[i].values if hasattr(X_shap, 'iloc') else X_shap[i]

    # Force plot
    shap.force_plot(
        base_value,
        sv[i],
        x_row,
        feature_names=FEATURE_COLS,
        matplotlib=True,
        show=False,
        figsize=(16, 3)
    )
    plt.title(f'Individual Prediction — Sample {i} (Predicted: DEFAULT)',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_force_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Red bars  → INCREASE default probability")
    print("Blue bars → DECREASE default probability")
else:
    print("No predicted defaulters in the first n_samples rows.")

 Final Summary & Business Insights


In [ ]:
#  FINAL SUMMARY
print("=" * 65)
print("         CREDIT RISK PREDICTION — FINAL RESULTS")
print("=" * 65)

print(f"\n Dataset:")
print(f"   Total samples  : {df.shape[0]:,}")
print(f"   Features used  : {len(FEATURE_COLS)} (including 8 engineered)")
print(f"   Default rate   : {y.mean()*100:.1f}%")
print(f"   Imbalance fix  : SMOTE applied")

print(f"\n Model Performance (Test Set):")
print(f"   {'Model':<22} {'ROC-AUC':>8} {'F1-Score':>9} {'Recall':>8}")
print(f"   {'─'*55}")
for name, res in sorted(results.items(),
                         key=lambda x: x[1]['ROC-AUC'], reverse=True):
    star = ' ' if name == best_model_name else ''
    print(f"   {name:<22} {res['ROC-AUC']:>8.4f} "
          f"{res['F1-Score']:>9.4f} {res['Recall']:>8.4f}{star}")

print(f"\n Best Model: {best_model_name}")
print(f"   ROC-AUC  = {results[best_model_name]['ROC-AUC']:.4f}")
print(f"   F1-Score = {results[best_model_name]['F1-Score']:.4f}")
print(f"   Recall   = {results[best_model_name]['Recall']:.4f}")

print(f"\n Key Risk Factors (from SHAP Analysis):")
risk_factors = [
    "RevolvingUtilizationOfUnsecuredLines — High credit card usage",
    "NumberOfTimes90DaysLate   — History of severe delays",
    "TotalLatePayments         — Overall payment behavior",
    "DebtRatio                 — Debt burden vs income",
    "MonthlyIncome             — Repayment capacity",
    "HasSevereDelinquency      — Ever missed 90+ days",
]
for i, factor in enumerate(risk_factors, 1):
    print(f"   {i}. {factor}")

print(f"\n Output Files Generated:")
files = [
    'class_distribution.png',
    'feature_distributions.png',
    'correlation_heatmap.png',
    'roc_curves.png',
    'model_comparison.png',
    'confusion_matrix.png',
    'shap_summary_beeswarm.png',
    'shap_bar_importance.png',
    'shap_force_plot.png',
]
for f in files:
    print(f"    {f}")

print("=" * 65)



*Project by Zunera | Credit Risk Prediction using ML | Give Me Some Credit Dataset*
